## Data Cleaning

#### Load data

In [102]:
import pandas as pd

listings = pd.read_csv("..\\processed\\CRMLSListing_Residential_With_Rates.csv", low_memory=False)
sold = pd.read_csv("..\\processed\\CRMLSSold_Residential_With_Rates.csv", low_memory=False)

#### Convert Data Types

Based on metadata information, convert columns to the correct intended data type.

In [103]:
def convert_dates(df):
    """
    Convert date columns to datetime format.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the date columns.
    
    Returns:
    pd.DataFrame: The DataFrame with converted date columns.
    """
    df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
    df["ListingContractDate"] = pd.to_datetime(df["ListingContractDate"], errors="coerce")
    df["PurchaseContractDate"] = pd.to_datetime(df["PurchaseContractDate"], errors="coerce")
    df["ContractStatusChangeDate"] = pd.to_datetime(df["ContractStatusChangeDate"], errors="coerce")
    
    return df

listings = convert_dates(listings)
sold = convert_dates(sold)

def convert_to_int64(df):
    """
    Convert columns to Int64 type.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the columns to convert.
    
    Returns:
    pd.DataFrame: The DataFrame with converted columns.
    """
    columns_to_convert = ["YearBuilt", "BathroomsTotalInteger", "BedroomsTotal", "Stories", "MainLevelBedrooms"]
    
    for column in columns_to_convert:
        df[column] = df[column].astype('Int64', errors="ignore")
    
    return df

listings = convert_to_int64(listings)
sold = convert_to_int64(sold)


Confirm data types after conversion:

In [104]:
print("LISTINGS ---------------------------")
print(pd.DataFrame({"Dtype": listings.dtypes}))

print("\nSOLD -------------------------------")
print(pd.DataFrame({"Dtype": sold.dtypes}))

LISTINGS ---------------------------
                            Dtype
OriginalListPrice         float64
ListingKey                  int64
ListAgentEmail             object
CloseDate          datetime64[ns]
ClosePrice                float64
...                           ...
AssociationFee            float64
LotSizeSquareFeet         float64
UnparsedAddress.1          object
year_month                 object
rate_30yr_fixed           float64

[73 rows x 1 columns]

SOLD -------------------------------
                                 Dtype
Flooring                        object
ViewYN                          object
WaterfrontYN                    object
BasementYN                      object
PoolPrivateYN                   object
...                                ...
BathroomsTotalInteger_Outlier     bool
DaysOnMarket_Outlier              bool
YearBuilt_Outlier                 bool
year_month                      object
rate_30yr_fixed                float64

[82 rows x 1 columns]


#### Flag Date Inconsistencies

CloseDate must occur on or after all other dates.

In [105]:
for name, df in [("Sold", sold), ("Listings", listings)]:
    df["listing_after_close_flag"] = df["ListingContractDate"] > df["CloseDate"]
    df["purchase_after_close_flag"] = df["PurchaseContractDate"] > df["CloseDate"]
    df["negative_timeline_flag"] = df["ListingContractDate"] > df["PurchaseContractDate"]

    print(f"\n{name} -------------------------------")
    print(f"Rows with listing_after_close_flag = True: {df['listing_after_close_flag'].sum()}")
    print(f"Rows with purchase_after_close_flag = True: {df['purchase_after_close_flag'].sum()}")
    print(f"Rows with negative_timeline_flag = True: {df['negative_timeline_flag'].sum()}\n")


Sold -------------------------------
Rows with listing_after_close_flag = True: 74
Rows with purchase_after_close_flag = True: 243
Rows with negative_timeline_flag = True: 311


Listings -------------------------------
Rows with listing_after_close_flag = True: 90
Rows with purchase_after_close_flag = True: 265
Rows with negative_timeline_flag = True: 320



#### Remove Invalid Rows

Rows with invalid values for numeric datapoints will be removed: ClosePrice $\le$ 0, LivingArea $\le$ 0, DaysOnMarket < 0,
negative Bedrooms or Bathrooms.

First, we can calculate valid DaysOnMarket for invalid rows if the row has valid date consistency.

In [106]:
# Identify rows with invalid DaysOnMarket and a False listing_after_close_flag
mask = (
    (sold["listing_after_close_flag"] == False) &
    (
        sold["DaysOnMarket"].isna() |
        (sold["DaysOnMarket"] < 0)
    ) &
    sold["ListingContractDate"].notna() &
    sold["CloseDate"].notna()
)

# Calculate actual days on market
sold.loc[mask, "DaysOnMarket"] = (
    sold.loc[mask, "CloseDate"] -
    sold.loc[mask, "ListingContractDate"]
).dt.days

print(f"Recalculated DaysOnMarket for {mask.sum()} rows")

sold.loc[
    mask,
    ["ListingContractDate", "CloseDate", "DaysOnMarket",
     "listing_after_close_flag"]
].head(10)

Recalculated DaysOnMarket for 54 rows


,ListingContractDate,CloseDate,DaysOnMarket,listing_after_close_flag
4,2023-11-17,2024-01-22,66,False
393,2024-01-08,2024-01-19,11,False
11205,2024-02-24,2024-02-26,2,False
11372,2024-02-16,2024-02-23,7,False
18689,2023-12-12,2024-02-05,55,False
22996,2023-09-25,2024-02-29,157,False
24388,2024-01-30,2024-03-21,51,False
24396,2024-01-11,2024-03-19,68,False
24432,2024-03-03,2024-03-15,12,False
29416,2024-02-07,2024-03-08,30,False


Remove remaining invalid rows: negative close prices, living areas, days on market, bedrooms, and bathroom counts

In [107]:
for name, df in [("Sold", sold), ("Listings", listings)]:
    print(f"{name}: before filtering, {len(df)} rows")

    invalid_conditions = {
        "ClosePrice": df["ClosePrice"].notna() & (df["ClosePrice"] <= 0),
        "LivingArea": df["LivingArea"].notna() & (df["LivingArea"] <= 0),
        "DaysOnMarket": df["DaysOnMarket"].notna() & (df["DaysOnMarket"] < 0),
        "BedroomsTotal": df["BedroomsTotal"].notna() & (df["BedroomsTotal"] < 0),
        "BathroomsTotalInteger": (
            df["BathroomsTotalInteger"].notna()
            & (df["BathroomsTotalInteger"] < 0)
        ),
    }
    invalid_null_counts = pd.DataFrame({
        "Invalid Rows": {
            column: condition.sum()
            for column, condition in invalid_conditions.items()
        },
        "Null Rows": {
            column: df[column].isna().sum()
            for column in invalid_conditions
        },
    })
    print(f"\n{name} invalid and null row counts:")
    print(invalid_null_counts.to_string())

    mask = (
        (df["ClosePrice"].isna() | (df["ClosePrice"] > 0)) &
        (df["LivingArea"].isna() | (df["LivingArea"] > 0)) &
        (df["DaysOnMarket"].isna() | (df["DaysOnMarket"] >= 0)) &
        (df["BedroomsTotal"].isna() | (df["BedroomsTotal"] >= 0)) &
        (df["BathroomsTotalInteger"].isna() | (df["BathroomsTotalInteger"] >= 0))
    )

    filtered_df = df[mask].copy()

    print(f"{name}: after filtering, {len(filtered_df)} rows")

    if name == "Sold":
        sold = filtered_df
    else:
        listings = filtered_df

Sold: before filtering, 480487 rows

Sold invalid and null row counts:
                       Invalid Rows  Null Rows
ClosePrice                        1          2
LivingArea                      172        269
DaysOnMarket                      0          0
BedroomsTotal                     0         12
BathroomsTotalInteger             0         70
Sold: after filtering, 480314 rows
Listings: before filtering, 644458 rows

Listings invalid and null row counts:
                       Invalid Rows  Null Rows
ClosePrice                        0     451293
LivingArea                      422        621
DaysOnMarket                     34          0
BedroomsTotal                     0        165
BathroomsTotalInteger             0         56
Listings: after filtering, 644002 rows


#### Remove Columns

For our market and competitive analysis needs, we can keep only a small subset of the original columns as a separate dataset.

In [108]:
market_cols = [
    # Identifiers
    "ListingKey", "ListingKeyNumeric", "ListingId",

    # Location
    "PostalCode", "City", "CountyOrParish",
    "StateOrProvince", "MLSAreaMajor", 
    "Latitude", "Longitude", "UnparsedAddress",

    # Pricing and dates
    "ListPrice", "OriginalListPrice", "ClosePrice",
    "CloseDate", "ListingContractDate",
    "PurchaseContractDate", "ContractStatusChangeDate",
    "DaysOnMarket", "MlsStatus",

    # Property characteristics
    "PropertyType", "PropertySubType", "LivingArea",
    "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt",
    "LotSizeAcres", "LotSizeArea", "LotSizeSquareFeet",
    "Stories", "Levels", "GarageSpaces", "ParkingTotal",
    "FireplaceYN", "PoolPrivateYN", "ViewYN",
    "WaterfrontYN", "BasementYN", "AttachedGarageYN",
    "NewConstructionYN", "AssociationFee",

    # Competition
    "ListAgentFullName", "ListAgentFirstName",
    "ListAgentLastName", "ListAgentAOR", "ListOfficeName",
    "BuyerAgentMlsId", "BuyerAgentFirstName",
    "BuyerAgentLastName", "BuyerAgentAOR", "BuyerOfficeName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "CoListOfficeName"
]

sold_clean = sold[
    [col for col in market_cols if col in sold.columns]
].copy()

listings_clean = listings[
    [col for col in market_cols if col in listings.columns]
].copy()

print(f"Sold before vs. after shape: {sold.shape} vs. {sold_clean.shape}")
print(f"Listings before vs. after shape: {listings.shape} vs. {listings_clean.shape}")

Sold before vs. after shape: (480314, 85) vs. (480314, 54)
Listings before vs. after shape: (644002, 76) vs. (644002, 48)


Check null counts of remaining columns:

In [109]:
list_null = pd.DataFrame({
    'Dtype': listings_clean.dtypes,
    'Null Count': listings_clean.isnull().sum(),
    'Null Pct (%)': (listings_clean.isnull().sum() / len(listings_clean)) * 100
})

print("LISTINGS -------------------------------------------------------")
print(list_null)

sold_null = pd.DataFrame({
    'Dtype': sold_clean.dtypes,
    'Null Count': sold_clean.isnull().sum(),
    'Null Pct (%)': (sold_clean.isnull().sum() / len(sold_clean)) * 100
})

print("\nSOLD -----------------------------------------------------------")
print(sold_null)

LISTINGS -------------------------------------------------------
                                   Dtype  Null Count  Null Pct (%)
ListingKey                         int64           0      0.000000
ListingKeyNumeric                  int64           0      0.000000
ListingId                         object           0      0.000000
PostalCode                        object          20      0.003106
City                              object         604      0.093789
CountyOrParish                    object           0      0.000000
StateOrProvince                   object          71      0.011025
MLSAreaMajor                      object       87866     13.643746
Latitude                         float64       81183     12.606017
Longitude                        float64       81183     12.606017
UnparsedAddress                   object         968      0.150310
ListPrice                        float64           0      0.000000
OriginalListPrice                float64         931      0.1445

#### Handle Missing Values

Out of the remaining columns, WaterfrontYN and BasementYN requires handling. Null values are converted to "False" value with the assumption that data was not entered if the binary was false; all existing values are "True" values.

In [110]:
sold_clean["BasementYN"] = sold_clean["BasementYN"].fillna(False)
sold_clean["WaterfrontYN"] = sold_clean["WaterfrontYN"].fillna(False)

print(sold_clean["BasementYN"].unique())
print(sold_clean["WaterfrontYN"].unique())

[False  True]
[False  True]


C:\Users\tiffa\AppData\Local\Temp\ipykernel_7396\1196776816.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sold_clean["BasementYN"] = sold_clean["BasementYN"].fillna(False)
C:\Users\tiffa\AppData\Local\Temp\ipykernel_7396\1196776816.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sold_clean["WaterfrontYN"] = sold_clean["WaterfrontYN"].fillna(False)


#### Geographic Features

Flag counts prior to cleaning

In [111]:
for name, df in [("Sold", sold_clean), ("Listings", listings_clean)]:
    print(f"{name} --------------------")
    print(f"Missing latitude: {df["Latitude"].isnull().sum()}")
    print(f"Missing longitude: {df["Longitude"].isnull().sum()}")

    print(f"Zero latitude: {(df["Latitude"] == 0).sum()}")
    print(f"Zero longitude: {(df["Longitude"] == 0).sum()}")

    print(f"Positive longitude (CA): {((df["Longitude"] > 0) & (df["StateOrProvince"] == "CA")).sum()}")

    print(f"Out of state latitude (CA): {((df["StateOrProvince"] == "CA") & ((df["Latitude"] < 32.5) | (df["Latitude"] > 42.1))).sum()}")
    print(f"Out of state longitude (CA): {((df["StateOrProvince"] == "CA") & (df["Longitude"] < -124.5) | (df["Longitude"] > -114)).sum()}")

    print(f"Impossible latitude: {((df["Latitude"].notna()) & (~df["Latitude"].between(-90, 90))).sum()}")
    print(f"Impossible longitude: {((df["Longitude"].notna()) & (~df["Longitude"].between(-180, 180))).sum()}")

Sold --------------------
Missing latitude: 4675
Missing longitude: 4675
Zero latitude: 42
Zero longitude: 42
Positive longitude (CA): 35
Out of state latitude (CA): 61
Out of state longitude (CA): 100
Impossible latitude: 1
Impossible longitude: 1
Listings --------------------
Missing latitude: 81183
Missing longitude: 81183
Zero latitude: 82
Zero longitude: 82
Positive longitude (CA): 63
Out of state latitude (CA): 138
Out of state longitude (CA): 300
Impossible latitude: 5
Impossible longitude: 1


1. Cleaning state and county codes

In [112]:
# Check inconsistencies and states: Sold
states = sold_clean["StateOrProvince"].unique()
print(states)

for state in states[1:]:
    print(f"\nState: {state}")
    print(sold_clean[sold_clean["StateOrProvince"] == state]["UnparsedAddress"])
    print(sold_clean[sold_clean["StateOrProvince"] == state]["CountyOrParish"])

['CA' 'NV' 'AZ' 'ID' 'TX' 'OS' 'GA' 'ME' 'NY' 'CO' 'MO' 'TN' 'FL' 'AR'
 'VA' nan]

State: NV
4827      571 W Painted Trails Road
167343            571 Junebug Place
Name: UnparsedAddress, dtype: object
4827      Other State
167343    Other State
Name: CountyOrParish, dtype: object

State: AZ
32071      2259 Wildflower Street
56232     15450 W MIRAMONTE Court
126120       3649 S Benson Avenue
141023      2022 Merrill Lane 10E
196581      7300 N Snow Bowl Road
198294           1000 VISTA Drive
291954            1628 Neptune Dr
301216     14444 W Moccasin Trail
333962          2581 N Hwy 95 533
408090         1057 Terrace Drive
Name: UnparsedAddress, dtype: object
32071     Other County
56232      Other State
126120    Other County
141023     Other State
196581           Other
198294     Other State
291954     Other State
301216           Other
333962     Other State
408090     Other State
Name: CountyOrParish, dtype: object

State: ID
39822    301 N 1st St 804 Coeur d'Alene
Name: Unparse

In [113]:
# Check inconsistencies and states: Listings
states = listings_clean["StateOrProvince"].unique()
print(states)

for state in states[1:]:
    print(f"\nState: {state}")
    print(listings_clean[listings_clean["StateOrProvince"] == state]["UnparsedAddress"])
    print(listings_clean[listings_clean["StateOrProvince"] == state]["CountyOrParish"])

['CA' 'AZ' 'TX' nan 'OS' 'FL' 'GA' 'MD' 'SC' 'CO' 'NY' 'ME' 'MO' 'NV' 'OR'
 'BC' 'WA' 'TN' 'DC' 'HI' 'PA' 'IL' 'MT' 'LA' 'NJ' 'VA' 'NC']

State: AZ
2533          2259 Wildflower Street
32137             805 S Sycamore 132
41973           3649 S Benson Avenue
130252         2581 N Highway 95 195
143990         7300 N Snow Bowl Road
165459              1231 Redwall Way
168303    12935 N Desert Olive Drive
239647        14444 W Moccasin Trail
247880              1000 VISTA Drive
257068           3260 Fan Palm Drive
286924        911 E Lone Pine Circle
303442         6300 N Camino Miraval
368382               1628 Neptune Dr
390240           24370 S 184Th Place
440939             2581 N Hwy 95 533
504276            1057 Terrace Drive
512413         6300 N Camino Miraval
Name: UnparsedAddress, dtype: object
2533      Other County
32137      Other State
41973     Other County
130252     Other State
143990    Other County
165459    Other County
168303    Other County
239647           Other
24

In [114]:
# Correct state codes confirmed from the address, city, and postal code.
state_corrections = [
    ("ME", "1805 49th Street", "92102", "CA", "San Diego"),
    ("AR", "1631 Neptune Avenue", "92024", "CA", "San Diego"),
    ("VA", "3253 Verde Ct", "94588", "CA", "Alameda"),
    ("DC", "17264 libertad Street", "92127", "CA", "San Diego"),
    ("FL", "2407 Telegraph Ave. 504", "94612", "CA", "Alameda"),
    ("OS", "302 CAMINO SAN CLEMENTE", "92672", "CA", "Orange"),
    ("OS", "184 Marsella", "92173", "CA", "San Diego"),
    ("OS", "10444 CANOGA Avenue 39", "91311", "CA", "Los Angeles"),
    ("CA", "1135 N Tin Whip Trail", "86314", "AZ", "Other state"),
    ("CA", "2022 Merrill Lane 10E", "86442", "AZ", "Other state"),
    ("CA", "1065 Blue Sands Court", "89011", "NV", "Other state"),
    ("CA", "5196 Golden Lane L", "89119", "NV", "Other state"),
    ("CA", "3353 W 75th S", "74132", "OK", "Other state"),
    ("CA", "550 E 54th Street N", "74126", "OK", "Other state"),
    ("CA", "64645 MAMALAHOA HWY", "96743", "HI", "Other state"),
    ("CA", "6413 W Bloomfield Road", "85304", "AZ", "Other state"),
    ("CA", "19717 SW Mt Bachelor Drive 231", "97702", "OR", "Other state"),
    ("CA", "17186 Highway 126", "97759", "OR", "Other state")
    ]

# Correct county codes from confirmed address and ZIP code
county_corrections = [
    ("1566 Karock Rd", "95223", "Calaveras"),
    ("168 Norman ave", "94520-1105", "Contra Costa"),
    ("7387 Carrollton Pl", "88888", "Riverside"),
    ("24214 El Pilar #73", "92677", "Orange"),
    ("2883 Chinook Way", "95223", "Calaveras"),
    ("2883 2883 Chinook Way", "95223", "Calaveras"),
    ("13148 Bent Grass Place", "93021", "Ventura"),
    ("3646 Muriettas Roost", "95223", "Calaveras"),
    ("18850 Newville Road", "95963", "Tehama"),
    ("10550 Wilshire Boulevard 1204", "90024", "Los Angeles"),
    ("558 Marlin ct", "94065", "San Mateo"),
    ("2048 Woodland Dr", "93222", "Kern"),
    ("4348 Mammoth Avenue 203", "91423", "Los Angeles"),
    ("10041 Oso Avenue", "91311", "Los Angeles"),
    ("333 Leonard Ave", "93529", "Mono")
    ]

data_frames = [sold_clean, listings_clean]
for df in data_frames:
    addresses = df["UnparsedAddress"].astype("string").str.strip().str.casefold()
    postal_codes = (
        df["PostalCode"].astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    states = df["StateOrProvince"].astype("string").str.strip().str.upper()

    # State correction
    for old_state, address, postal_code, new_state, county in state_corrections:
        mask = (
            states.eq(old_state)
            & addresses.eq(address.casefold())
            & postal_codes.eq(postal_code)
        ).fillna(False)
        df.loc[mask, "StateOrProvince"] = new_state
        df.loc[mask, "CountyOrParish"] = county

    # County correction
    states = df["StateOrProvince"].astype("string").str.strip().str.upper()
    for address, postal_code, county in county_corrections:
        mask = (
            states.eq("CA")
            & addresses.eq(address.casefold())
            & postal_codes.eq(postal_code)
        ).fillna(False)
        df.loc[mask, "CountyOrParish"] = county

    # Sign correction of CA coordinates that are positive & clearly signed wrong
    positive_longitude = (
        states.eq("CA")
        & df["Latitude"].between(30, 43)
        & df["Longitude"].between(114, 125, inclusive="left")
    ).fillna(False)
    df.loc[positive_longitude, "Longitude"] *= -1

    # Country correction
    df.loc[states.eq("OS").fillna(False), "CountyOrParish"] = "Foreign Country"
    non_ca = states.notna() & ~states.isin(["CA", "OS"])
    df.loc[non_ca, "CountyOrParish"] = "Other state"

print("Corrected state counts:")
print(sold_clean["StateOrProvince"].value_counts(dropna=False).to_string())
print("\nListings:")
print(listings_clean["StateOrProvince"].value_counts(dropna=False).to_string())

Corrected state counts:
StateOrProvince
CA     480285
AZ         11
NV          3
OS          3
CO          2
FL          2
NaN         2
ID          1
TX          1
GA          1
NY          1
MO          1
TN          1

Listings:
StateOrProvince
CA     643770
OS         85
NaN        71
AZ         20
OR          9
NV          6
FL          5
BC          4
TX          4
HI          4
CO          3
NY          3
TN          3
OK          2
MO          2
PA          1
NJ          1
LA          1
MT          1
IL          1
GA          1
WA          1
MD          1
ME          1
SC          1
NC          1


2. Flag coordinate inconsistencies

In [115]:
def oos(df):
    """
    Identify out-of-state (OOS) properties based on latitude and longitude.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the property data.
    
    Returns:
    pd.DataFrame: The DataFrame with additional columns indicating OOS properties.
    """
    state_bounds = {
        "CA": {
            "lat_min": 32.5,
            "lat_max": 42.1,
            "long_min": -124.5,
            "long_max": -114.0
        }, 
        "NV": {
            "lat_min": 35.0,
            "lat_max": 42.0,
            "long_min": -120.0,
            "long_max": -114.0
        },
        "AZ": {
            "lat_min": 31.3,
            "lat_max": 37.0,
            "long_min": -114.8,
            "long_max": -109.0
        },
        "ID": {
            "lat_min": 42.0,
            "lat_max": 49.0,
            "long_min": -117.0,
            "long_max": -111.0
        },
        "TX": {
            "lat_min": 25.8,
            "lat_max": 36.5,
            "long_min": -106.6,
            "long_max": -93.5
        },
        "GA": {
            "lat_min": 30.3,
            "lat_max": 35.1,
            "long_min": -85.6,
            "long_max": -80.8
        },
        "ME": {
            "lat_min": 44.0,
            "lat_max": 47.5,
            "long_min": -71.0,
            "long_max": -69.0
        },
        "NY": {
            "lat_min": 40.5,
            "lat_max": 45.0,
            "long_min": -79.8,
            "long_max": -71.8
        },
        "CO": {
            "lat_min": 36.9,
            "lat_max": 41.0,
            "long_min": -109.0,
            "long_max": -102.0
        },
        "MO": {
            "lat_min": 36.0,
            "lat_max": 40.8,
            "long_min": -95.8,
            "long_max": -89.1
        },
        "TN": {
            "lat_min": 34.9,
            "lat_max": 36.6,
            "long_min": -90.4,
            "long_max": -81.6
        },
        "FL": {
            "lat_min": 24.5,
            "lat_max": 31.0,
            "long_min": -87.6,
            "long_max": -80.1
        },
        "AR": {
            "lat_min": 33.0,
            "lat_max": 36.5,
            "long_min": -94.6,
            "long_max": -89.6
        },
        "VA": {
            "lat_min": 36.5,
            "lat_max": 39.5,
            "long_min": -83.7,
            "long_max": -75.2
        },
        "KS": {
            "lat_min": 36.9,
            "lat_max": 40.0,
            "long_min": -102.1,
            "long_max": -94.6
        },
        "BC": {
            "lat_min": 28.0,
            "lat_max": 33.2,
            "long_min": -117.2,
            "long_max": -112.7
        },
        "DC": {
            "lat_min": 38.7,
            "lat_max": 39.1,
            "long_min": -77.2,
            "long_max": -76.8
        },
        "HI": {
            "lat_min": 18.8,
            "lat_max": 22.3,
            "long_min": -160.3,
            "long_max": -154.7
        },
        "IL": {
            "lat_min": 36.9,
            "lat_max": 42.6,
            "long_min": -91.6,
            "long_max": -87.0
        },
        "LA": {
            "lat_min": 28.9,
            "lat_max": 33.1,
            "long_min": -94.1,
            "long_max": -89.0
        },
        "MD": {
            "lat_min": 37.9,
            "lat_max": 39.8,
            "long_min": -79.5,
            "long_max": -75.0
        },
        "MT": {
            "lat_min": 44.3,
            "lat_max": 49.1,
            "long_min": -116.1,
            "long_max": -104.0
        },
        "NC": {
            "lat_min": 33.8,
            "lat_max": 36.6,
            "long_min": -84.4,
            "long_max": -75.4
        },
        "NJ": {
            "lat_min": 38.9,
            "lat_max": 41.4,
            "long_min": -75.6,
            "long_max": -73.9
        },
        "OR": {
            "lat_min": 41.9,
            "lat_max": 46.3,
            "long_min": -124.7,
            "long_max": -116.4
        },
        "PA": {
            "lat_min": 39.7,
            "lat_max": 42.3,
            "long_min": -80.6,
            "long_max": -74.7
        },
        "SC": {
            "lat_min": 32.0,
            "lat_max": 35.3,
            "long_min": -83.4,
            "long_max": -78.5
        },
        "WA": {
            "lat_min": 45.5,
            "lat_max": 49.1,
            "long_min": -124.9,
            "long_max": -116.9
        },
        "OK": {
            "lat_min": 33.6,
            "lat_max": 37.1,
            "long_min": -103.0,
            "long_max": -94.4
        }
    }

    states = df["StateOrProvince"].astype("string").str.strip().str.upper()
    unknown_states = set(states.dropna().unique()) - set(state_bounds) - {"OS"}
    if unknown_states:
        raise ValueError(f"Missing coordinate bounds for states: {sorted(unknown_states)}")

    df["oos_lat"] = False
    df["oos_long"] = False

    # Flag out of state in context of each state
    for state, bounds in state_bounds.items():
        mask = states.eq(state).fillna(False)
        df.loc[mask, "oos_lat"] = (
            df.loc[mask, "Latitude"].notna()
            & ~df.loc[mask, "Latitude"].between(bounds["lat_min"], bounds["lat_max"])
        )
        df.loc[mask, "oos_long"] = (
            df.loc[mask, "Longitude"].notna()
            & ~df.loc[mask, "Longitude"].between(bounds["long_min"], bounds["long_max"])
        )

    return df

In [116]:
# Add flags to dataset
for name, df in [("Sold", sold_clean), ("Listings", listings_clean)]:
    df["missing_lat"] = df["Latitude"].isnull()
    df["missing_long"] = df["Longitude"].isnull()

    df["zero_lat"] = df["Latitude"] == 0
    df["zero_long"] = df["Longitude"] == 0

    df["positive_long_CA"] = (df["Longitude"] > 0) & (df["StateOrProvince"] == "CA")

    oos(df)

    df["impossible_lat"] = df["Latitude"].notna() & ~df["Latitude"].between(-90, 90)
    df["impossible_long"] = df["Longitude"].notna() & ~df["Longitude"].between(-180, 180)

geo_flags = [
    "missing_lat",
    "missing_long",
    "zero_lat",
    "zero_long",
    "positive_long_CA",
    "oos_lat",
    "oos_long",
    "impossible_lat",
    "impossible_long"
]

for name, df in [("Sold", sold_clean), ("Listings", listings_clean)]:
    summary = pd.DataFrame({
        "Flagged_Count": df[geo_flags].sum()
    })

    print(f"\n{'=' * 15} {name} Geographic Checks {'=' * 15}")
    print(summary)


=============== Sold Geographic Checks ===============
                  Flagged_Count
missing_lat                4675
missing_long               4675
zero_lat                     42
zero_long                    42
positive_long_CA              6
oos_lat                      61
oos_long                     57
impossible_lat                1
impossible_long               1

=============== Listings Geographic Checks ===============
                  Flagged_Count
missing_lat               81183
missing_long              81183
zero_lat                     82
zero_long                    82
positive_long_CA             11
oos_lat                     138
oos_long                    107
impossible_lat                5
impossible_long               1


In [117]:
print(sold_clean[sold_clean["positive_long_CA"] == 1][["StateOrProvince", "UnparsedAddress", "CountyOrParish", "Latitude", "Longitude"]])
print(listings_clean[listings_clean["positive_long_CA"] == 1][["StateOrProvince", "UnparsedAddress", "CountyOrParish", "Latitude", "Longitude"]])

       StateOrProvince       UnparsedAddress  CountyOrParish  Latitude  \
9714                CA    1202 W Pine Street          Orange       1.0   
62742               CA   1999 Highland Drive          Merced      50.0   
147137              CA     3227 E 3rd Street     Los Angeles      40.0   
213957              CA  17824 Pitache Street  San Bernardino      20.0   
268699              CA       1416 Opal Court       Riverside       1.0   
468882              CA           2767 Sonoma     Los Angeles      10.0   

        Longitude  
9714          2.0  
62742       100.0  
147137      125.0  
213957      329.0  
268699        1.0  
468882       12.0  
       StateOrProvince         UnparsedAddress  CountyOrParish  Latitude  \
55921               CA    15321 San Ardo Drive     Los Angeles     737.0   
69468               CA     1999 Highland Drive          Merced      50.0   
151283              CA      1174 Hampton Place       Riverside      33.0   
183649              CA  321 W Queen S

#### Value Cleaning: Standardization

In [118]:
# Clean up white space and empty strings

for df in [sold_clean, listings_clean]:
    # Clean text columns without changing capitalization yet
    text_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in text_cols:
        df[col] = df[col].map(
            lambda value: value.strip() if isinstance(value, str) else value
        )

        # Convert empty strings to missing values
        df[col] = df[col].replace("", pd.NA)

In [119]:
print(sold_clean["PostalCode"].unique())

# Standardize PostalCode
for df in [sold_clean, listings_clean]:
    if "PostalCode" in df.columns:
        df["PostalCode"] = (
            df["PostalCode"]
            .astype("string")
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

        # Pad valid four-digit ZIP strings with a leading zero
        valid_zip = df["PostalCode"].str.fullmatch(r"\d{1,5}", na=False)

        df.loc[valid_zip, "PostalCode"] = (
            df.loc[valid_zip, "PostalCode"].str.zfill(5)
        )

print(sold_clean["PostalCode"].unique())

['94401' '91950' '93401' ... '94546-1013' '92435' '95134-2613']
<StringArray>
[     '94401',      '91950',      '93401',      '95437',      '95148',
      '92101',      '90026',      '90065',      '94030',      '94403',
 ...
 '94595-3164', '94563-3348', '95391-1183', '94520-2901', '95655-3071',
 '95350-8920', '95351-3754', '94546-1013',      '92435', '95134-2613']
Length: 4051, dtype: string


In [120]:
# Standardize boolean columns to True/False values

boolean_cols = [
    "FireplaceYN",
    "AttachedGarageYN",
    "NewConstructionYN",
    "PoolPrivateYN",
    "ViewYN",
    "WaterfrontYN",
    "BasementYN",
]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False,
}

for df in [sold_clean, listings_clean]:
    for col in boolean_cols:
        if col in df.columns:
            normalized = df[col].astype("string").str.strip().str.lower()
            mapped = normalized.map(bool_map)

            # Preserve unknown or missing values as <NA>
            df[col] = mapped.astype("boolean")

In [121]:
# Standardize capitalization
title_case_cols = [
    "City",
    "CountyOrParish"
]

uppercase_cols = [
    "MlsStatus",
    "PropertyType",
    "PropertySubType",
    "PostalCode",
]

for df in [sold_clean, listings_clean]:
    for col in title_case_cols:
        if col in df.columns:
            df[col] = df[col].str.title()

    for col in uppercase_cols:
        if col in df.columns:
            df[col] = df[col].str.upper()

Validate cleaning:

In [122]:
for name, df in [("sold", sold_clean), ("listings", listings_clean)]:
    print(f"\n{'=' * 15} {name.upper()} {'=' * 15}")

    # Duplicate identifiers
    for col in ["ListingKey", "ListingId"]:
        if col in df.columns:
            print(f"Duplicate {col}: {df[col].duplicated().sum():,}")

    # Impossible or suspicious values
    checks = {
        "Nonpositive ListPrice": (
            "ListPrice", lambda s: s.notna() & (s <= 0)
        ),
        "Nonpositive ClosePrice": (
            "ClosePrice", lambda s: s.notna() & (s <= 0)
        ),
        "Nonpositive LivingArea": (
            "LivingArea", lambda s: s.notna() & (s <= 0)
        ),
        "Negative DaysOnMarket": (
            "DaysOnMarket", lambda s: s.notna() & (s < 0)
        ),
        "Future YearBuilt": (
            "YearBuilt", lambda s: s.notna() & (s > 2026)
        ),
    }

    for label, (col, condition) in checks.items():
        if col in df.columns:
            print(f"{label}: {condition(df[col]).sum():,}")


=============== SOLD ===============
Duplicate ListingKey: 378
Duplicate ListingId: 380
Nonpositive ListPrice: 0
Nonpositive ClosePrice: 0
Nonpositive LivingArea: 0
Negative DaysOnMarket: 0
Future YearBuilt: 2

=============== LISTINGS ===============
Duplicate ListingKey: 201
Duplicate ListingId: 224
Nonpositive ListPrice: 0
Nonpositive ClosePrice: 0
Nonpositive LivingArea: 0
Negative DaysOnMarket: 0
Future YearBuilt: 82


Listings with future YearBuilt are valid as they could be listed and sold before completion.

Check differences between duplicate listings: appears that CloseDates are different, likely fallen through

In [123]:
for name, df in [("Sold", sold_clean), ("Listings", listings_clean)]:
    print(f"\n{name} duplicate identifier pairs:")
    detail_cols = [
        col for col in [
            "ListingKey", "ListingId", "UnparsedAddress", "City",
            "PostalCode", "StateOrProvince", "ListingContractDate", 
            "CloseDate", "MlsStatus"
        ]
        if col in df.columns
    ]

    for col in ["ListingKey", "ListingId"]:
        if col not in df.columns:
            continue

        duplicate_mask = df[col].notna() & df[col].duplicated(keep=False)
        duplicate_rows = df.loc[duplicate_mask, detail_cols].copy()
        duplicate_rows.insert(0, "Row", duplicate_rows.index)
        duplicate_rows["_identifier_sort"] = duplicate_rows[col].astype("string")
        duplicate_rows = duplicate_rows.sort_values(
            ["_identifier_sort", "Row"]
        ).drop(columns="_identifier_sort")

        duplicate_groups = df.loc[duplicate_mask].groupby(col).size()
        pair_count = int((duplicate_groups * (duplicate_groups - 1) // 2).sum())
        print(
            f"{col}: {len(duplicate_groups):,} duplicate identifiers, "
            f"{pair_count:,} row pairs"
        )
        print(duplicate_rows.to_string(index=False))


Sold duplicate identifier pairs:
ListingKey: 374 duplicate identifiers, 382 row pairs
   Row  ListingKey    ListingId                 UnparsedAddress                   City PostalCode StateOrProvince ListingContractDate  CloseDate MlsStatus
 92410  1031992525   SW22259219              998 Shelbie Avenue                Brawley      92227              CA          2022-12-22 2024-06-21    CLOSED
157856  1031992525   SW22259219              998 Shelbie Avenue                Brawley      92227              CA          2022-12-22 2024-10-18    CLOSED
 75722  1037464932   PW23082927               13448 Park Street               Whittier      90601              CA          2023-05-13 2024-05-25    CLOSED
141571  1037464932   PW23082927               13448 Park Street               Whittier      90601              CA          2023-05-13 2024-09-06    CLOSED
 75680  1038376137   OC23108406         33558 Sea Gull Court 59             Dana Point      92629              CA          2023-06-14 2024

### Save filtered data

In [131]:
sold_clean.to_csv("..\\processed\\sold_final.csv")
listings_clean.to_csv("..\\processed\\listings_final.csv")